In [1]:
from google.colab import files
import pandas as pd
import sqlite3
import numpy as np
from datetime import datetime

uploaded = files.upload()

# Getting the filename
filename = list(uploaded.keys())[0]
print(f"\n✅ File '{filename}' uploaded successfully!")
print(f"   File size: {len(uploaded[filename]) / (1024*1024):.2f} MB")

📁 Please click 'Choose Files' and select your CSV file
   (online_retail.csv or data.csv)



Saving Online_Retail.csv to Online_Retail.csv

✅ File 'Online_Retail.csv' uploaded successfully!
   File size: 41.92 MB


---
## STEP 2: Load and Inspect the Data

**This cell will:**
- Load the CSV file into memory
- Shows few rows & column information

In [2]:
# Loading the dataset
print("📊 Loading dataset...")
df = pd.read_csv(filename, encoding='ISO-8859-1')

print(f"\n✅ Dataset loaded successfully!")
print(f"   Total rows: {len(df):,}")
print(f"   Total columns: {len(df.columns)}")

print("\n📋 Column names:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i}. {col}")

print("\n👀 First 5 rows:")
print(df.head())

print("\n📈 Data types:")
print(df.dtypes)

print("\n🔍 Missing values:")
print(df.isnull().sum())

📊 Loading dataset...

✅ Dataset loaded successfully!
   Total rows: 541,909
   Total columns: 8

📋 Column names:
   1. InvoiceNo
   2. StockCode
   3. Description
   4. Quantity
   5. InvoiceDate
   6. UnitPrice
   7. CustomerID
   8. Country

👀 First 5 rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

    InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/10 8:26       2.55     17850.0  United Kingdom  
1  12/1/10 8:26       3.39     17850.0  United Kingdom  
2  12/1/10 8:26       2.75     17850.0  United Kingdom  
3  12/1/10 8:26       3.39     17850.0  United Kingdom  
4  12/1/10 8:26       

---
## STEP 3: Clean the Data

**Data cleaning steps:**
1. Removing rows with missing CustomerID
2. Removing cancelled orders (InvoiceNo starting with 'C')
3. Removing negative/zero quantities
4. Removing zero/negative prices
5. Converting dates to proper format

In [3]:
print("🧹 Cleaning data...\n")
print(f"Original dataset: {len(df):,} rows")

# Step 1: Remove missing CustomerID
df_clean = df.dropna(subset=['CustomerID'])
print(f"After removing missing CustomerID: {len(df_clean):,} rows ({len(df)-len(df_clean):,} removed)")

# Step 2: Remove cancelled orders
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
print(f"After removing cancelled orders: {len(df_clean):,} rows")

# Step 3: Remove negative or zero quantities
df_clean = df_clean[df_clean['Quantity'] > 0]
print(f"After removing invalid quantities: {len(df_clean):,} rows")

# Step 4: Remove zero or negative prices
df_clean = df_clean[df_clean['UnitPrice'] > 0]
print(f"After removing invalid prices: {len(df_clean):,} rows")

# Step 5: Convert InvoiceDate to datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
print(f"\n✅ Date conversion completed")

# Display cleaned data summary
print(f"\n📊 Final cleaned dataset: {len(df_clean):,} rows")
print(f"   Data retention rate: {100*len(df_clean)/len(df):.1f}%")

print("\n👀 Sample of cleaned data:")
print(df_clean.head())

🧹 Cleaning data...

Original dataset: 541,909 rows
After removing missing CustomerID: 406,829 rows (135,080 removed)
After removing cancelled orders: 397,924 rows
After removing invalid quantities: 397,924 rows
After removing invalid prices: 397,884 rows


/tmp/ipython-input-2386879136.py:21: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])



✅ Date conversion completed

📊 Final cleaned dataset: 397,884 rows
   Data retention rate: 73.4%

👀 Sample of cleaned data:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


---
## STEP 4: Add Calculated Columns

**New columns created:**
- TotalPrice: Quantity × UnitPrice
- Year, Month: Extracted from date
- YearMonth: Combined year-month (e.g., "2011-01")
- Date: Simple date format

In [4]:
print("➕ Adding calculated columns...\n")

# Calculate total price
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']
print("✅ Added: TotalPrice")

# Extract year and month
df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
print("✅ Added: Year, Month")

# Create YearMonth string
df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M').astype(str)
print("✅ Added: YearMonth")

# Create simple date column
df_clean['Date'] = df_clean['InvoiceDate'].dt.date
print("✅ Added: Date")

print("\n📋 Updated column list:")
for i, col in enumerate(df_clean.columns, 1):
    print(f"   {i}. {col}")

print("\n👀 Sample with new columns:")
print(df_clean[['InvoiceNo', 'Quantity', 'UnitPrice', 'TotalPrice', 'YearMonth', 'Date']].head(10))

➕ Adding calculated columns...

✅ Added: TotalPrice
✅ Added: Year, Month
✅ Added: YearMonth
✅ Added: Date

📋 Updated column list:
   1. InvoiceNo
   2. StockCode
   3. Description
   4. Quantity
   5. InvoiceDate
   6. UnitPrice
   7. CustomerID
   8. Country
   9. TotalPrice
   10. Year
   11. Month
   12. YearMonth
   13. Date

👀 Sample with new columns:
  InvoiceNo  Quantity  UnitPrice  TotalPrice YearMonth        Date
0    536365         6       2.55       15.30   2010-12  2010-12-01
1    536365         6       3.39       20.34   2010-12  2010-12-01
2    536365         8       2.75       22.00   2010-12  2010-12-01
3    536365         6       3.39       20.34   2010-12  2010-12-01
4    536365         6       3.39       20.34   2010-12  2010-12-01
5    536365         2       7.65       15.30   2010-12  2010-12-01
6    536365         6       4.25       25.50   2010-12  2010-12-01
7    536366         6       1.85       11.10   2010-12  2010-12-01
8    536366         6       1.85      

---
## STEP 5: Quick Data Analysis

**Before creating the database, let's see some quick insights:**

In [5]:
print("📊 QUICK DATA INSIGHTS\n")
print("=" * 60)

# Basic statistics
print("\n📈 Dataset Overview:")
print(f"   Total transactions: {len(df_clean):,}")
print(f"   Unique orders: {df_clean['InvoiceNo'].nunique():,}")
print(f"   Unique customers: {df_clean['CustomerID'].nunique():,}")
print(f"   Unique products: {df_clean['StockCode'].nunique():,}")
print(f"   Countries: {df_clean['Country'].nunique()}")

# Date range
print("\n📅 Time Period:")
print(f"   From: {df_clean['Date'].min()}")
print(f"   To: {df_clean['Date'].max()}")
print(f"   Duration: {(df_clean['Date'].max() - df_clean['Date'].min()).days} days")

# Revenue statistics
total_revenue = df_clean['TotalPrice'].sum()
print("\n💰 Revenue Statistics:")
print(f"   Total revenue: £{total_revenue:,.2f}")
print(f"   Average transaction: £{df_clean['TotalPrice'].mean():.2f}")
print(f"   Median transaction: £{df_clean['TotalPrice'].median():.2f}")
print(f"   Largest transaction: £{df_clean['TotalPrice'].max():,.2f}")

# Top 5 countries
print("\n🌍 Top 5 Countries by Revenue:")
top_countries = df_clean.groupby('Country')['TotalPrice'].sum().sort_values(ascending=False).head(5)
for i, (country, revenue) in enumerate(top_countries.items(), 1):
    pct = 100 * revenue / total_revenue
    print(f"   {i}. {country}: £{revenue:,.2f} ({pct:.1f}%)")

# Top 5 products
print("\n🏆 Top 5 Products by Revenue:")
top_products = df_clean.groupby('Description')['TotalPrice'].sum().sort_values(ascending=False).head(5)
for i, (product, revenue) in enumerate(top_products.items(), 1):
    print(f"   {i}. {product[:50]}: £{revenue:,.2f}")

print("\n" + "=" * 60)

📊 QUICK DATA INSIGHTS


📈 Dataset Overview:
   Total transactions: 397,884
   Unique orders: 18,532
   Unique customers: 4,338
   Unique products: 3,665
   Countries: 37

📅 Time Period:
   From: 2010-12-01
   To: 2011-12-09
   Duration: 373 days

💰 Revenue Statistics:
   Total revenue: £8,911,407.90
   Average transaction: £22.40
   Median transaction: £11.80
   Largest transaction: £168,469.60

🌍 Top 5 Countries by Revenue:
   1. United Kingdom: £7,308,391.55 (82.0%)
   2. Netherlands: £285,446.34 (3.2%)
   3. EIRE: £265,545.90 (3.0%)
   4. Germany: £228,867.14 (2.6%)
   5. France: £209,024.05 (2.3%)

🏆 Top 5 Products by Revenue:
   1. PAPER CRAFT , LITTLE BIRDIE: £168,469.60
   2. REGENCY CAKESTAND 3 TIER: £142,592.95
   3. WHITE HANGING HEART T-LIGHT HOLDER: £100,448.15
   4. JUMBO BAG RED RETROSPOT: £85,220.78
   5. MEDIUM CERAMIC TOP STORAGE JAR: £81,416.73



---
## STEP 6: Create SQLite Database

- Creates a SQLite database file
- Saves all your cleaned data
- This file will be used in DB Browser for SQLite

In [6]:
print("💾 Creating SQLite database...\n")

# Create database connection
db_filename = 'online_retail.db'
conn = sqlite3.connect(db_filename)
print(f"✅ Database connection created: {db_filename}")

# Write dataframe to SQLite
print(f"\n📤 Writing {len(df_clean):,} rows to database...")
df_clean.to_sql('retail_data', conn, if_exists='replace', index=False)
print("✅ Data written successfully!")

# Verify the data was loaded correctly
print("\n🔍 Verifying database...")
cursor = conn.cursor()

# Check row count
cursor.execute("SELECT COUNT(*) FROM retail_data")
row_count = cursor.fetchone()[0]
print(f"   ✅ Row count: {row_count:,}")

# Check column count
cursor.execute("PRAGMA table_info(retail_data)")
columns = cursor.fetchall()
print(f"   ✅ Column count: {len(columns)}")

# Display column names
print("\n   📋 Columns in database:")
for col in columns:
    print(f"      - {col[1]} ({col[2]})")

# Test a simple query
print("\n🧪 Testing a sample query...")
test_query = "SELECT COUNT(*), SUM(TotalPrice) as total_revenue FROM retail_data"
cursor.execute(test_query)
result = cursor.fetchone()
print(f"   Total rows: {result[0]:,}")
print(f"   Total revenue: £{result[1]:,.2f}")

# Close connection
conn.close()

print("\n✅ Database created successfully!")
print(f"\n📁 File name: {db_filename}")
print(f"   Table name: retail_data")
print(f"   Ready to download in next step!")

💾 Creating SQLite database...

✅ Database connection created: online_retail.db

📤 Writing 397,884 rows to database...
✅ Data written successfully!

🔍 Verifying database...
   ✅ Row count: 397,884
   ✅ Column count: 13

   📋 Columns in database:
      - InvoiceNo (TEXT)
      - StockCode (TEXT)
      - Description (TEXT)
      - Quantity (INTEGER)
      - InvoiceDate (TIMESTAMP)
      - UnitPrice (REAL)
      - CustomerID (REAL)
      - Country (TEXT)
      - TotalPrice (REAL)
      - Year (INTEGER)
      - Month (INTEGER)
      - YearMonth (TEXT)
      - Date (DATE)

🧪 Testing a sample query...
   Total rows: 397,884
   Total revenue: £8,911,407.90

✅ Database created successfully!

📁 File name: online_retail.db
   Table name: retail_data
   Ready to download in next step!


---
## STEP 7: Download the Database

In [7]:
import os

# Check file size
file_size = os.path.getsize(db_filename) / (1024 * 1024)  # Convert to MB

print("📦 Preparing download...\n")
print(f"   File name: {db_filename}")
print(f"   File size: {file_size:.2f} MB")
print(f"   Table: retail_data")
print(f"   Rows: {row_count:,}")

print("\n⬇️ Starting download...")
files.download(db_filename)

print("\n✅ Download initiated!")
print("\n📋 NEXT STEPS:")
print("   1. Save the file to: Desktop/DataProjects/Ecommerce_Analysis/database/")
print("   2. Open DB Browser for SQLite")
print("   3. Click 'Open Database' and select this file")
print("   4. You can now run SQL queries!")
print("\n🎉 Great job! Your database is ready to use!")

📦 Preparing download...

   File name: online_retail.db
   File size: 49.57 MB
   Table: retail_data
   Rows: 397,884

⬇️ Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download initiated!

📋 NEXT STEPS:
   1. Save the file to: Desktop/DataProjects/Ecommerce_Analysis/database/
   2. Open DB Browser for SQLite
   3. Click 'Open Database' and select this file
   4. You can now run SQL queries!

🎉 Great job! Your database is ready to use!
